# XGBoost2 Top Feature Removal Robustness Check

This notebook removes the top XGBoost feature and retrains the model to generate `xgboost2_remove_top_feature_metrics.csv` for Section 5.4.

In [1]:
# ============================================================
# XGBoost2 Top-Feature Removal Robustness Check
#
# 目的：
#   为报告 Section 5.4 生成：
#   xgboost2_remove_top_feature_metrics.csv
#
# 做法：
#   1. 读取原来的 stage4_final_75_features_panel.parquet
#   2. 读取 XGBoost2 实际使用的 75 个特征
#   3. 从中删除 top feature: r_id_mean_5_lag1_cs_rank
#   4. 使用原 XGBoost2 最优参数重新训练模型
#   5. 输出 daily IC、long-short Sharpe、full vs no-top comparison
#
# 注意：
#   这不是重新做完整调参，而是为了满足：
#   "Removing your top feature should not collapse the Sharpe."
#   因此应保持模型参数、训练目标、sample split 等设定不变，只删除一个特征。
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor


# ============================================================
# 0. 基本设置
# ============================================================

RANDOM_STATE = 42

# 原始收益率目标变量
TARGET_COL = "target_r_on_next_winsor"

# 真实收益率列，用于 long-short 检验
REALIZED_RETURN_COL_CANDIDATE = "target_r_on_next"

# 训练目标模式：和原 XGBoost2 保持一致
MODEL_TARGET_MODE = "cs_rank"

# 最终模型是否用 train + validation 重训
# 为了和原 notebook 保持一致，默认 False
REFIT_FINAL_ON_TRAIN_VALID = False

# 要删除的 top feature
# 如果原 XGBoost2 输出文件 xgboost2_feature_importance_gain.csv 存在，代码会优先读取其中排名第一的 feature
DEFAULT_TOP_FEATURE_TO_REMOVE = "r_id_mean_5_lag1_cs_rank"

# 原 XGBoost2 notebook 中选出的最优参数：model_id = 20
# 如果原输出目录中存在 xgboost2_run_config.json，代码会优先读取其中的 best_params
FALLBACK_BEST_PARAMS = {
    "n_estimators": 400,
    "max_depth": 5,
    "learning_rate": 0.02,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "min_child_weight": 10,
    "reg_lambda": 10,
    "reg_alpha": 0.1,
    "gamma": 0.1,
}


# ============================================================
# 1. 自动定位项目目录与输出目录
# ============================================================

def find_project_dir():
    """
    自动寻找 MLF_coursework 项目根目录。
    适用于：
    1. 当前 notebook / py 文件在 MLF_coursework 根目录下；
    2. 当前 notebook / py 文件在 MLF_coursework 子文件夹下；
    3. 当前目录下可以搜索到 stage4_final_75_features_panel.parquet。
    """
    cwd = Path.cwd().resolve()

    for p in [cwd] + list(cwd.parents):
        if p.name == "MLF_coursework":
            return p

    for p in [cwd] + list(cwd.parents):
        if (p / "stage4_final_75_features_panel.parquet").exists():
            return p

    matches = list(cwd.rglob("stage4_final_75_features_panel.parquet"))
    if len(matches) > 0:
        return matches[0].parent

    raise FileNotFoundError(
        "没有找到 MLF_coursework 项目目录，也没有找到 stage4_final_75_features_panel.parquet。"
    )


PROJECT_DIR = find_project_dir()

DATA_PATH = PROJECT_DIR / "stage4_final_75_features_panel.parquet"

FEATURE_SELECTION_DIR = PROJECT_DIR / "stage4_feature_selection_outputs"

# 原 XGBoost2 输出目录
FULL_MODEL_OUTPUT_DIR = FEATURE_SELECTION_DIR / "XGBoost2_prediction_outcome"

# 这次 no-top-feature 检验的输出目录
OUTPUT_DIR = FEATURE_SELECTION_DIR / "XGBoost2_remove_top_feature_outcome"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 可能存在的特征文件
BORUTA_FEATURE_FILE = FEATURE_SELECTION_DIR / "boruta75_feature_cols.txt"
XGB_USED_FEATURE_FILE = FULL_MODEL_OUTPUT_DIR / "xgboost2_feature_list_used.csv"
FULL_IMPORTANCE_FILE = FULL_MODEL_OUTPUT_DIR / "xgboost2_feature_importance_gain.csv"
FULL_RUN_CONFIG_FILE = FULL_MODEL_OUTPUT_DIR / "xgboost2_run_config.json"

print("=" * 100)
print("Project directory:")
print(PROJECT_DIR)

print("\nInput data path:")
print(DATA_PATH)

print("\nFull model output directory:")
print(FULL_MODEL_OUTPUT_DIR)

print("\nNo-top-feature output directory:")
print(OUTPUT_DIR)
print("=" * 100)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"找不到输入数据文件：{DATA_PATH}")


# ============================================================
# 2. 读取原 XGBoost2 的 top feature 和 best parameters
# ============================================================

def load_top_feature_to_remove():
    """
    优先从原 full XGBoost2 的 feature importance 文件中读取排名第一的 feature。
    如果文件不存在，则使用 DEFAULT_TOP_FEATURE_TO_REMOVE。
    """
    if FULL_IMPORTANCE_FILE.exists():
        imp = pd.read_csv(FULL_IMPORTANCE_FILE)
        if "feature" in imp.columns and len(imp) > 0:
            top_feature = str(imp.loc[0, "feature"])
            print("\nTop feature read from full-model importance file:")
            print(top_feature)
            return top_feature

    print("\nFull-model feature importance file not found or invalid.")
    print("Using default top feature:")
    print(DEFAULT_TOP_FEATURE_TO_REMOVE)
    return DEFAULT_TOP_FEATURE_TO_REMOVE


def load_best_params():
    """
    优先从原 full XGBoost2 的 run_config.json 读取 best_params。
    如果文件不存在，则使用 notebook 记录的第 20 组最优参数。
    """
    if FULL_RUN_CONFIG_FILE.exists():
        with open(FULL_RUN_CONFIG_FILE, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        if "best_params" in cfg and isinstance(cfg["best_params"], dict):
            print("\nBest params read from full-model run_config.json:")
            print(cfg["best_params"])
            return cfg["best_params"]

    print("\nFull-model run_config.json not found or invalid.")
    print("Using fallback best params from original notebook:")
    print(FALLBACK_BEST_PARAMS)
    return FALLBACK_BEST_PARAMS


TOP_FEATURE_TO_REMOVE = load_top_feature_to_remove()
BEST_PARAMS = load_best_params()


# ============================================================
# 3. 读取数据
# ============================================================

df = pd.read_parquet(DATA_PATH)

print("\nOriginal data shape:", df.shape)

if "date" not in df.columns:
    raise ValueError("数据中找不到 date 列。")

df["date"] = pd.to_datetime(df["date"])

sort_cols = ["date"]
if "ticker" in df.columns:
    sort_cols.append("ticker")
elif "instrument_id" in df.columns:
    sort_cols.append("instrument_id")

df = df.sort_values(sort_cols).reset_index(drop=True)

print("Date range:", df["date"].min(), "to", df["date"].max())
print("Number of columns:", len(df.columns))


# ============================================================
# 4. 只保留可交易样本
# ============================================================

if "is_trade_eligible" in df.columns:
    before_rows = len(df)
    df = df[df["is_trade_eligible"].fillna(0).astype(bool)].copy()
    after_rows = len(df)

    print("\nFiltered by is_trade_eligible == True")
    print("Rows before:", before_rows)
    print("Rows after:", after_rows)
else:
    print("\nNo is_trade_eligible column found. Using all rows.")


# ============================================================
# 5. 设置目标变量
# ============================================================

if TARGET_COL not in df.columns:
    raise ValueError(f"找不到目标变量 {TARGET_COL}。")

df = df[df[TARGET_COL].notna()].copy()

# 每日横截面 rank target
df["target_cs_rank"] = (
    df.groupby("date")[TARGET_COL]
      .rank(method="average", pct=True)
      - 0.5
)

# 每日横截面 z-score target
df["target_cs_zscore"] = (
    df.groupby("date")[TARGET_COL]
      .transform(lambda x: (x - x.mean()) / (x.std(ddof=0) + 1e-12))
)

if MODEL_TARGET_MODE == "cs_rank":
    MODEL_TARGET_COL = "target_cs_rank"
elif MODEL_TARGET_MODE == "raw_return":
    MODEL_TARGET_COL = TARGET_COL
else:
    raise ValueError("MODEL_TARGET_MODE 只能是 'cs_rank' 或 'raw_return'。")

df = df[df[MODEL_TARGET_COL].notna()].copy()

if REALIZED_RETURN_COL_CANDIDATE in df.columns:
    REALIZED_RETURN_COL = REALIZED_RETURN_COL_CANDIDATE
else:
    REALIZED_RETURN_COL = TARGET_COL

print("\nOriginal return target:", TARGET_COL)
print("Model training target:", MODEL_TARGET_COL)
print("Model target mode:", MODEL_TARGET_MODE)
print("Realized return column for long-short:", REALIZED_RETURN_COL)
print("Data shape after target cleaning:", df.shape)


# ============================================================
# 6. 读取原 XGBoost2 使用的 75 个特征，并删除 top feature
# ============================================================

def read_feature_lines_txt(path):
    raw_lines = path.read_text(encoding="utf-8").splitlines()

    features = []
    for line in raw_lines:
        line = line.strip()
        line = line.replace(",", "")
        line = line.replace('"', "")
        line = line.replace("'", "")

        if line == "":
            continue
        if line.lower() in ["feature", "features", "column", "columns"]:
            continue

        features.append(line)

    return features


def infer_numeric_feature_cols(data):
    exclude_cols = {
        "date",
        "ticker",
        "instrument_id",
        "sample_split",
        "is_trade_eligible",
        "target_r_on_next",
        "target_r_on_next_winsor",
        "target_cs_rank",
        "target_cs_zscore",
    }

    numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()

    features = []
    for col in numeric_cols:
        col_lower = col.lower()

        if col in exclude_cols:
            continue
        if "target" in col_lower:
            continue
        if "future" in col_lower:
            continue
        if "next" in col_lower:
            continue
        if "lead" in col_lower:
            continue
        if "pred" in col_lower:
            continue
        if "score" in col_lower:
            continue

        features.append(col)

    return features


def load_feature_list(data):
    """
    优先级：
    1. 原 full XGBoost2 实际使用的 xgboost2_feature_list_used.csv
    2. Boruta 75 feature list: boruta75_feature_cols.txt
    3. 从 parquet 中自动推断数值型特征
    """
    if XGB_USED_FEATURE_FILE.exists():
        print("\nUsing full XGBoost2 used feature list from:")
        print(XGB_USED_FEATURE_FILE)
        feature_df = pd.read_csv(XGB_USED_FEATURE_FILE)

        if "feature" not in feature_df.columns:
            raise ValueError(f"{XGB_USED_FEATURE_FILE} 中找不到 feature 列。")

        return feature_df["feature"].astype(str).tolist()

    if BORUTA_FEATURE_FILE.exists():
        print("\nUsing Boruta 75 feature list from:")
        print(BORUTA_FEATURE_FILE)
        return read_feature_lines_txt(BORUTA_FEATURE_FILE)

    print("\nNo feature list file found.")
    print("Automatically inferring numeric feature columns from the dataset.")
    return infer_numeric_feature_cols(data)


feature_cols_full = load_feature_list(df)

# 清理不存在或非数值的特征
missing_features = [c for c in feature_cols_full if c not in df.columns]
if len(missing_features) > 0:
    print("\nWarning: these features are in the feature list but not in the dataset:")
    for c in missing_features[:50]:
        print(" -", c)

feature_cols_full = [c for c in feature_cols_full if c in df.columns]

non_numeric_features = [
    c for c in feature_cols_full
    if not pd.api.types.is_numeric_dtype(df[c])
]

if len(non_numeric_features) > 0:
    print("\nWarning: removing non-numeric features:")
    for c in non_numeric_features:
        print(" -", c)

feature_cols_full = [c for c in feature_cols_full if c not in non_numeric_features]

if TOP_FEATURE_TO_REMOVE not in feature_cols_full:
    raise ValueError(
        f"要删除的 top feature {TOP_FEATURE_TO_REMOVE} 不在当前特征列表中。"
        f"请检查 {XGB_USED_FEATURE_FILE} 或 {FULL_IMPORTANCE_FILE}。"
    )

feature_cols_no_top = [c for c in feature_cols_full if c != TOP_FEATURE_TO_REMOVE]

print("\nOriginal number of full XGBoost2 features:", len(feature_cols_full))
print("Removed top feature:", TOP_FEATURE_TO_REMOVE)
print("Number of features after removal:", len(feature_cols_no_top))

if len(feature_cols_no_top) == 0:
    raise ValueError("删除 top feature 后没有剩余特征。")

pd.DataFrame({"feature": feature_cols_full}).to_csv(
    OUTPUT_DIR / "xgboost2_full_feature_list_reference.csv",
    index=False,
)

pd.DataFrame({"feature": feature_cols_no_top}).to_csv(
    OUTPUT_DIR / "xgboost2_no_top_feature_list_used.csv",
    index=False,
)


# ============================================================
# 7. 划分 train / validation / test
# ============================================================

def build_train_valid_test_masks(data):
    """
    优先使用 sample_split。
    如果没有 sample_split，则按日期做 60% / 20% / 20% 时间切分。
    """
    if "sample_split" in data.columns:
        split = data["sample_split"].astype(str).str.lower()

        train_mask = split.isin(["train", "training"])
        valid_mask = split.isin(["valid", "validation", "val"])
        test_mask = split.isin(["test", "oos", "out_of_sample"])

        if train_mask.sum() > 0 and valid_mask.sum() > 0 and test_mask.sum() > 0:
            print("\nUsing existing sample_split column.")
            return train_mask, valid_mask, test_mask

        print("\nsample_split exists but cannot identify train / validation / test.")
        print("Creating time-based split instead.")

    unique_dates = np.array(sorted(data["date"].unique()))
    n_dates = len(unique_dates)

    train_end = int(n_dates * 0.60)
    valid_end = int(n_dates * 0.80)

    train_dates = unique_dates[:train_end]
    valid_dates = unique_dates[train_end:valid_end]
    test_dates = unique_dates[valid_end:]

    train_mask = data["date"].isin(train_dates)
    valid_mask = data["date"].isin(valid_dates)
    test_mask = data["date"].isin(test_dates)

    data["sample_split"] = "unused"
    data.loc[train_mask, "sample_split"] = "train"
    data.loc[valid_mask, "sample_split"] = "validation"
    data.loc[test_mask, "sample_split"] = "test"

    print("\nCreated time-based split by date.")

    return train_mask, valid_mask, test_mask


train_mask, valid_mask, test_mask = build_train_valid_test_masks(df)

train_index = df.index[train_mask].to_numpy()
valid_index = df.index[valid_mask].to_numpy()
test_index = df.index[test_mask].to_numpy()

print("\nSplit summary:")
print("Train rows:", len(train_index))
print("Validation rows:", len(valid_index))
print("Test rows:", len(test_index))

print("\nTrain date range:", df.loc[train_mask, "date"].min(), "to", df.loc[train_mask, "date"].max())
print("Valid date range:", df.loc[valid_mask, "date"].min(), "to", df.loc[valid_mask, "date"].max())
print("Test date range:", df.loc[test_mask, "date"].min(), "to", df.loc[test_mask, "date"].max())


# ============================================================
# 8. 评价函数
# ============================================================

def regression_metrics(y_true, y_pred):
    """
    常规回归指标。
    如果 MODEL_TARGET_MODE = 'cs_rank'，这里评价的是 rank target 的拟合。
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) == 0:
        return {
            "RMSE": np.nan,
            "MAE": np.nan,
            "R2": np.nan,
            "Pearson_IC": np.nan,
            "Spearman_IC": np.nan,
        }

    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)

    try:
        r2 = r2_score(y_true, y_pred)
    except Exception:
        r2 = np.nan

    pearson_ic = pd.Series(y_true).corr(pd.Series(y_pred), method="pearson")
    spearman_ic = pd.Series(y_true).corr(pd.Series(y_pred), method="spearman")

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Pearson_IC": pearson_ic,
        "Spearman_IC": spearman_ic,
    }


def daily_ic_table(data, y_col, pred_col):
    """
    每天横截面计算 Pearson IC 和 Spearman IC。
    """
    rows = []

    for date, g in data.groupby("date"):
        g = g[[y_col, pred_col]].dropna()

        if len(g) < 5:
            continue

        pearson_ic = g[y_col].corr(g[pred_col], method="pearson")
        spearman_ic = g[y_col].corr(g[pred_col], method="spearman")

        rows.append({
            "date": date,
            "n_stocks": len(g),
            "daily_pearson_ic": pearson_ic,
            "daily_spearman_ic": spearman_ic,
        })

    return pd.DataFrame(rows)


def daily_ic_summary_from_table(daily_ic):
    """
    对 daily IC 做汇总，并额外计算 t-stat。
    """
    if len(daily_ic) == 0:
        return {
            "mean_daily_pearson_ic": np.nan,
            "mean_daily_spearman_ic": np.nan,
            "std_daily_spearman_ic": np.nan,
            "n_days": 0,
            "ic_ir": np.nan,
            "daily_spearman_ic_tstat": np.nan,
        }

    mean_p = daily_ic["daily_pearson_ic"].mean()
    mean_s = daily_ic["daily_spearman_ic"].mean()
    std_s = daily_ic["daily_spearman_ic"].std(ddof=1)
    n_days = daily_ic["date"].nunique()

    ic_ir = mean_s / std_s if std_s != 0 else np.nan
    tstat = mean_s / (std_s / np.sqrt(n_days)) if std_s != 0 and n_days > 1 else np.nan

    return {
        "mean_daily_pearson_ic": mean_p,
        "mean_daily_spearman_ic": mean_s,
        "std_daily_spearman_ic": std_s,
        "n_days": n_days,
        "ic_ir": ic_ir,
        "daily_spearman_ic_tstat": tstat,
    }


def long_short_backtest(data, realized_return_col, pred_col, q=0.10):
    """
    简单 long-short 检验：
    每天按照预测分数排序，做多 top 10%，做空 bottom 10%。
    """
    rows = []

    for date, g in data.groupby("date"):
        g = g[[realized_return_col, pred_col]].dropna().copy()

        if len(g) < 20:
            continue

        k = max(int(len(g) * q), 1)

        g = g.sort_values(pred_col)

        short_return = g.head(k)[realized_return_col].mean()
        long_return = g.tail(k)[realized_return_col].mean()

        long_short_return = long_return - short_return

        rows.append({
            "date": date,
            "n_stocks": len(g),
            "n_long": k,
            "n_short": k,
            "long_leg_return": long_return,
            "short_leg_return": short_return,
            "long_short_return": long_short_return,
        })

    out = pd.DataFrame(rows)

    if len(out) > 0:
        out["cum_long_short_return"] = (1 + out["long_short_return"]).cumprod() - 1

    return out


def long_short_summary_from_table(ls_table):
    """
    long-short 日收益汇总。
    """
    if len(ls_table) == 0:
        return {
            "mean_daily_long_short_return": np.nan,
            "std_daily_long_short_return": np.nan,
            "annualized_return_approx": np.nan,
            "annualized_vol_approx": np.nan,
            "sharpe_approx": np.nan,
            "positive_day_ratio": np.nan,
            "n_days": 0,
        }

    mean_ret = ls_table["long_short_return"].mean()
    std_ret = ls_table["long_short_return"].std(ddof=1)
    ann_ret = mean_ret * 252
    ann_vol = std_ret * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol != 0 else np.nan
    pos_ratio = (ls_table["long_short_return"] > 0).mean()

    return {
        "mean_daily_long_short_return": mean_ret,
        "std_daily_long_short_return": std_ret,
        "annualized_return_approx": ann_ret,
        "annualized_vol_approx": ann_vol,
        "sharpe_approx": sharpe,
        "positive_day_ratio": pos_ratio,
        "n_days": ls_table["date"].nunique(),
    }


# ============================================================
# 9. 删除 top feature 后重新训练 XGBoost2
# ============================================================

if REFIT_FINAL_ON_TRAIN_VALID:
    final_train_index = np.sort(np.concatenate([train_index, valid_index]))
    final_train_description = "train_plus_validation"
else:
    final_train_index = train_index
    final_train_description = "train_only"

print("\n" + "=" * 100)
print("Training XGBoost2 without top feature")
print("Removed feature:", TOP_FEATURE_TO_REMOVE)
print("Final training sample:", final_train_description)
print("Final training rows:", len(final_train_index))
print("Number of features:", len(feature_cols_no_top))
print("Best params:")
print(BEST_PARAMS)
print("=" * 100)

X_train_final = df.loc[final_train_index, feature_cols_no_top].astype("float32")
y_train_final = df.loc[final_train_index, MODEL_TARGET_COL].astype("float32")

no_top_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    tree_method="hist",
    max_bin=256,
    n_jobs=-1,
    eval_metric="rmse",
    importance_type="gain",
    missing=np.nan,
    **BEST_PARAMS,
)

no_top_model.fit(
    X_train_final,
    y_train_final,
    verbose=False,
)

print("No-top-feature model training finished.")


# ============================================================
# 10. 生成 train / validation / test 预测
# ============================================================

PRED_COL = "pred_xgboost2_no_top"
RANK_COL = "score_xgboost2_no_top_rank"

df[PRED_COL] = np.nan

for split_name, idx in [
    ("train", train_index),
    ("validation", valid_index),
    ("test", test_index),
]:
    print(f"Predicting {split_name} set...")

    X_temp = df.loc[idx, feature_cols_no_top].astype("float32")
    df.loc[idx, PRED_COL] = no_top_model.predict(X_temp)

df[RANK_COL] = df.groupby("date")[PRED_COL].rank(
    method="average",
    pct=True,
)

print("\nPrediction finished.")
print(df[["date", TARGET_COL, MODEL_TARGET_COL, PRED_COL, RANK_COL]].head())


# ============================================================
# 11. 计算 no-top-feature 模型指标
# ============================================================

metrics_rows = []
daily_ic_all = []
long_short_all = []

split_items = [
    ("train", train_mask),
    ("validation", valid_mask),
    ("test", test_mask),
]

for split_name, mask in split_items:
    temp = df.loc[mask].copy()

    # 1. overall model target metrics
    model_metrics = regression_metrics(
        temp[MODEL_TARGET_COL],
        temp[PRED_COL],
    )

    pooled_return_pearson_ic = temp[TARGET_COL].corr(temp[PRED_COL], method="pearson")
    pooled_return_spearman_ic = temp[TARGET_COL].corr(temp[PRED_COL], method="spearman")

    # 2. daily IC
    daily_ic = daily_ic_table(
        data=temp,
        y_col=TARGET_COL,
        pred_col=PRED_COL,
    )
    daily_ic["split"] = split_name
    daily_ic_all.append(daily_ic)

    daily_ic_summary = daily_ic_summary_from_table(daily_ic)

    # 3. long-short
    ls = long_short_backtest(
        data=temp,
        realized_return_col=REALIZED_RETURN_COL,
        pred_col=PRED_COL,
        q=0.10,
    )
    ls["split"] = split_name
    long_short_all.append(ls)

    ls_summary = long_short_summary_from_table(ls)

    row = {
        "model_variant": "xgboost2_without_top_feature",
        "split": split_name,
        "removed_feature": TOP_FEATURE_TO_REMOVE,
        "n_features_original": len(feature_cols_full),
        "n_features_used": len(feature_cols_no_top),
        "model_target": MODEL_TARGET_COL,

        "model_target_RMSE": model_metrics["RMSE"],
        "model_target_MAE": model_metrics["MAE"],
        "model_target_R2": model_metrics["R2"],
        "model_target_Pearson_IC": model_metrics["Pearson_IC"],
        "model_target_Spearman_IC": model_metrics["Spearman_IC"],

        "pooled_return_Pearson_IC": pooled_return_pearson_ic,
        "pooled_return_Spearman_IC": pooled_return_spearman_ic,

        "mean_daily_pearson_ic": daily_ic_summary["mean_daily_pearson_ic"],
        "mean_daily_spearman_ic": daily_ic_summary["mean_daily_spearman_ic"],
        "std_daily_spearman_ic": daily_ic_summary["std_daily_spearman_ic"],
        "daily_spearman_ic_tstat": daily_ic_summary["daily_spearman_ic_tstat"],
        "ic_ir": daily_ic_summary["ic_ir"],
        "n_ic_days": daily_ic_summary["n_days"],

        "mean_daily_long_short_return": ls_summary["mean_daily_long_short_return"],
        "std_daily_long_short_return": ls_summary["std_daily_long_short_return"],
        "annualized_return_approx": ls_summary["annualized_return_approx"],
        "annualized_vol_approx": ls_summary["annualized_vol_approx"],
        "long_short_sharpe": ls_summary["sharpe_approx"],
        "positive_day_ratio": ls_summary["positive_day_ratio"],
        "n_long_short_days": ls_summary["n_days"],
    }

    metrics_rows.append(row)

no_top_metrics_df = pd.DataFrame(metrics_rows)

daily_ic_df = pd.concat(daily_ic_all, ignore_index=True)
long_short_df = pd.concat(long_short_all, ignore_index=True)

print("\n" + "=" * 100)
print("No-top-feature metrics:")
print(no_top_metrics_df)
print("=" * 100)


# ============================================================
# 12. 如果原 full XGBoost2 输出存在，合并成 full vs no-top 对比表
# ============================================================

def try_load_full_model_summary():
    """
    尝试读取原 full XGBoost2 的 daily IC summary 和 long-short summary。
    如果找不到，就返回 None。
    """
    full_daily_path = FULL_MODEL_OUTPUT_DIR / "xgboost2_daily_ic_summary.csv"
    full_ls_path = FULL_MODEL_OUTPUT_DIR / "xgboost2_long_short_summary.csv"

    if not full_daily_path.exists() or not full_ls_path.exists():
        print("\nOriginal full-model summary files not found.")
        print("The required no-top-feature metrics will still be saved.")
        return None

    full_daily = pd.read_csv(full_daily_path)
    full_ls = pd.read_csv(full_ls_path)

    full = pd.merge(
        full_daily,
        full_ls,
        on="split",
        how="outer",
        suffixes=("_ic", "_ls"),
    )

    # 兼容原文件中的列名
    rows = []
    for _, r in full.iterrows():
        split_name = r.get("split", np.nan)

        mean_s = r.get("mean_daily_spearman_ic", np.nan)
        std_s = r.get("std_daily_spearman_ic", np.nan)
        n_days = r.get("n_days_ic", r.get("n_days", np.nan))

        if pd.notna(std_s) and std_s != 0 and pd.notna(n_days) and n_days > 1:
            tstat = mean_s / (std_s / np.sqrt(n_days))
        else:
            tstat = np.nan

        row = {
            "model_variant": "full_xgboost2_existing_output",
            "split": split_name,
            "removed_feature": "none",
            "n_features_original": len(feature_cols_full),
            "n_features_used": len(feature_cols_full),
            "model_target": MODEL_TARGET_COL,

            "model_target_RMSE": np.nan,
            "model_target_MAE": np.nan,
            "model_target_R2": np.nan,
            "model_target_Pearson_IC": np.nan,
            "model_target_Spearman_IC": np.nan,

            "pooled_return_Pearson_IC": np.nan,
            "pooled_return_Spearman_IC": np.nan,

            "mean_daily_pearson_ic": r.get("mean_daily_pearson_ic", np.nan),
            "mean_daily_spearman_ic": mean_s,
            "std_daily_spearman_ic": std_s,
            "daily_spearman_ic_tstat": tstat,
            "ic_ir": r.get("ic_ir", np.nan),
            "n_ic_days": n_days,

            "mean_daily_long_short_return": r.get("mean_daily_long_short_return", np.nan),
            "std_daily_long_short_return": r.get("std_daily_long_short_return", np.nan),
            "annualized_return_approx": r.get("annualized_return_approx", np.nan),
            "annualized_vol_approx": r.get("annualized_vol_approx", np.nan),
            "long_short_sharpe": r.get("sharpe_approx", np.nan),
            "positive_day_ratio": r.get("positive_day_ratio", np.nan),
            "n_long_short_days": r.get("n_days_ls", r.get("n_days", np.nan)),
        }
        rows.append(row)

    return pd.DataFrame(rows)


full_metrics_df = try_load_full_model_summary()

if full_metrics_df is not None:
    robustness_metrics_df = pd.concat(
        [full_metrics_df, no_top_metrics_df],
        ignore_index=True,
    )

    # 额外计算同一 split 中 no-top 相比 full 的变化
    full_ref = (
        full_metrics_df
        .set_index("split")[["mean_daily_spearman_ic", "daily_spearman_ic_tstat", "long_short_sharpe"]]
        .rename(columns={
            "mean_daily_spearman_ic": "full_mean_daily_spearman_ic",
            "daily_spearman_ic_tstat": "full_daily_spearman_ic_tstat",
            "long_short_sharpe": "full_long_short_sharpe",
        })
    )

    robustness_metrics_df = robustness_metrics_df.merge(
        full_ref,
        left_on="split",
        right_index=True,
        how="left",
    )

    robustness_metrics_df["delta_mean_daily_spearman_ic_vs_full"] = (
        robustness_metrics_df["mean_daily_spearman_ic"]
        - robustness_metrics_df["full_mean_daily_spearman_ic"]
    )

    robustness_metrics_df["delta_long_short_sharpe_vs_full"] = (
        robustness_metrics_df["long_short_sharpe"]
        - robustness_metrics_df["full_long_short_sharpe"]
    )

else:
    robustness_metrics_df = no_top_metrics_df.copy()
    robustness_metrics_df["full_mean_daily_spearman_ic"] = np.nan
    robustness_metrics_df["full_daily_spearman_ic_tstat"] = np.nan
    robustness_metrics_df["full_long_short_sharpe"] = np.nan
    robustness_metrics_df["delta_mean_daily_spearman_ic_vs_full"] = np.nan
    robustness_metrics_df["delta_long_short_sharpe_vs_full"] = np.nan


# ============================================================
# 13. 保存所有输出文件
# ============================================================

# 最重要：报告 Section 5.4 要看的文件
robustness_metrics_path = OUTPUT_DIR / "xgboost2_remove_top_feature_metrics.csv"
robustness_metrics_df.to_csv(robustness_metrics_path, index=False)

# no-top-feature 单独指标
no_top_metrics_df.to_csv(
    OUTPUT_DIR / "xgboost2_no_top_feature_metrics_only.csv",
    index=False,
)

# daily IC
daily_ic_df.to_csv(
    OUTPUT_DIR / "xgboost2_remove_top_feature_daily_ic.csv",
    index=False,
)

# long-short daily returns
long_short_df.to_csv(
    OUTPUT_DIR / "xgboost2_remove_top_feature_long_short_daily_returns.csv",
    index=False,
)

# predictions
keep_cols = []
for col in [
    "date",
    "ticker",
    "instrument_id",
    "sample_split",
    "is_trade_eligible",
    "target_r_on_next",
    "target_r_on_next_winsor",
    "target_cs_rank",
    "target_cs_zscore",
    PRED_COL,
    RANK_COL,
]:
    if col in df.columns:
        keep_cols.append(col)

predictions_df = df[keep_cols].copy()

predictions_df.to_parquet(
    OUTPUT_DIR / "xgboost2_remove_top_feature_predictions_all.parquet",
    index=False,
)

test_predictions_df = predictions_df.loc[test_mask].copy()

test_predictions_df.to_parquet(
    OUTPUT_DIR / "xgboost2_remove_top_feature_predictions_test.parquet",
    index=False,
)

test_predictions_df.to_csv(
    OUTPUT_DIR / "xgboost2_remove_top_feature_predictions_test.csv",
    index=False,
)

# feature importance after removing top feature
importance_df = pd.DataFrame({
    "feature": feature_cols_no_top,
    "importance_gain_normalized": no_top_model.feature_importances_,
})

booster_score = no_top_model.get_booster().get_score(importance_type="gain")
importance_df["importance_gain_raw"] = importance_df["feature"].map(booster_score).fillna(0.0)

importance_df = importance_df.sort_values(
    "importance_gain_raw",
    ascending=False,
).reset_index(drop=True)

importance_df.to_csv(
    OUTPUT_DIR / "xgboost2_remove_top_feature_importance_gain.csv",
    index=False,
)

# model
with open(OUTPUT_DIR / "xgboost2_remove_top_feature_model.pkl", "wb") as f:
    pickle.dump(no_top_model, f)

# config
run_config = {
    "purpose": "top_feature_removal_robustness_check_for_section_5_4",
    "target_col": TARGET_COL,
    "realized_return_col": REALIZED_RETURN_COL,
    "model_target_mode": MODEL_TARGET_MODE,
    "model_target_col": MODEL_TARGET_COL,
    "random_state": RANDOM_STATE,
    "refit_final_on_train_valid": REFIT_FINAL_ON_TRAIN_VALID,
    "final_train_description": final_train_description,
    "removed_feature": TOP_FEATURE_TO_REMOVE,
    "n_features_original": len(feature_cols_full),
    "n_features_used": len(feature_cols_no_top),
    "best_params": BEST_PARAMS,
    "input_data_path": str(DATA_PATH),
    "full_model_output_dir": str(FULL_MODEL_OUTPUT_DIR),
    "output_dir": str(OUTPUT_DIR),
}

with open(OUTPUT_DIR / "xgboost2_remove_top_feature_run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=4)

print("\n" + "=" * 100)
print("Top-feature removal robustness check finished.")
print("Most important output file:")
print(robustness_metrics_path)

print("\nGenerated files:")
for file in sorted(OUTPUT_DIR.iterdir()):
    print(" -", file.name)

print("\nTest split rows in xgboost2_remove_top_feature_metrics.csv:")
print(
    robustness_metrics_df.loc[
        robustness_metrics_df["split"].astype(str).str.lower().eq("test"),
        [
            "model_variant",
            "split",
            "removed_feature",
            "n_features_used",
            "mean_daily_spearman_ic",
            "daily_spearman_ic_tstat",
            "long_short_sharpe",
            "delta_long_short_sharpe_vs_full",
        ],
    ]
)

print("=" * 100)

Project directory:
C:\Users\yuchao\Desktop\MLF_coursework

Input data path:
C:\Users\yuchao\Desktop\MLF_coursework\stage4_final_75_features_panel.parquet

Full model output directory:
C:\Users\yuchao\Desktop\MLF_coursework\stage4_feature_selection_outputs\XGBoost2_prediction_outcome

No-top-feature output directory:
C:\Users\yuchao\Desktop\MLF_coursework\stage4_feature_selection_outputs\XGBoost2_remove_top_feature_outcome

Full-model feature importance file not found or invalid.
Using default top feature:
r_id_mean_5_lag1_cs_rank

Full-model run_config.json not found or invalid.
Using fallback best params from original notebook:
{'n_estimators': 400, 'max_depth': 5, 'learning_rate': 0.02, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 10, 'reg_lambda': 10, 'reg_alpha': 0.1, 'gamma': 0.1}

Original data shape: (3773971, 83)
Date range: 2010-01-04 00:00:00 to 2024-12-31 00:00:00
Number of columns: 83

Filtered by is_trade_eligible == True
Rows before: 3773971
Rows after: 